# Notebook 12 — Capstone: Oxford Flowers-102

Welcome to the first capstone. This notebook pulls together **every single technique** we have learned
across notebooks 01–11 and applies them to a real fine-grained classification benchmark:
[Oxford Flowers-102](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/).

Flowers-102 is a classic test of fine-grained visual recognition. It has:

- **102 classes** of flower species (roughly 3× more classes than Oxford Pets).
- **~8,189 images total**, split into `train`/`val`/`test` by torchvision.
- **Fine-grained confusions** — many lilies look like other lilies; many daisies look alike.
- **Modest sample counts per class** (40–258 per class), so augmentation and regularisation matter.

Our concrete target: **>90% top-1 accuracy in ~20 epochs** on the held-out test split, using
an `EfficientNetV2-S` backbone pre-trained on ImageNet-21k and fine-tuned on ImageNet-1k.

## Learning goals

By the end of this notebook you will have executed, end-to-end:

1. Downloading a real benchmark through torchvision.
2. Combining the provided splits and re-splitting *stratified* by class.
3. Inspecting class balance and sample diversity.
4. A production-flavoured augmentation pipeline (RandAugment, RandomErasing, Mixup).
5. Label smoothing as a form of soft regularisation.
6. Discriminative learning rates (backbone vs head) with warmup + cosine annealing.
7. An exponential moving average (EMA) of the weights for a free accuracy bump.
8. A rich evaluation: top-1, top-5, per-class F1, confusion matrix, per-class ROC, TTA.
9. Grad-CAM sanity checks on correctly- and incorrectly-classified test images.
10. Saving a final checkpoint suitable for downstream deployment or distillation.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/12_capstone_oxford_flowers.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

In [ ]:
# Imports — torch, torchvision, timm, sklearn metrics, and our utils package.
import copy
import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import transforms, datasets
from torchvision.transforms import RandAugment
from sklearn.model_selection import train_test_split
from sklearn.metrics import top_k_accuracy_score, f1_score, roc_auc_score
import timm

from utils.env import data_dir, checkpoints_dir
from utils.plotting import (
    plot_curves, plot_confusion_matrix, show_image_grid, plot_roc_pr,
)
from utils.metrics import classification_report_dict
from utils.training import evaluate, fit
from utils.gradcam import GradCAM, overlay_cam, find_efficientnet_target_layer

torch.backends.cudnn.benchmark = True  # fast convs for fixed input size
print('torch', torch.__version__, '| timm', timm.__version__, '| device', device)

## 2. Download Oxford Flowers-102

Torchvision ships a convenient wrapper. The first call downloads ~350 MB of images plus the labels;
subsequent calls just read from disk. We'll download all three pre-defined splits (`train`/`val`/`test`)
separately and then combine them — this gives us full control over how we re-split.

**Why re-split?** The official Flowers-102 `train` split only has **10 images per class**, which is
pedagogically cute but practically limiting. Combining the three splits gives us ~8k images that we
can re-partition 70/15/15 with our own random seed, stratified by class.

In [ ]:
root = data_dir()
print('data root:', root)

# First load with a *stub* transform so the raw PIL images are accessible.
# We'll wrap the split subsets with real augmentation pipelines later.
identity = transforms.Lambda(lambda x: x)

flowers_train_orig = datasets.Flowers102(root=root, split='train', download=True, transform=identity)
flowers_val_orig   = datasets.Flowers102(root=root, split='val',   download=True, transform=identity)
flowers_test_orig  = datasets.Flowers102(root=root, split='test',  download=True, transform=identity)

print('official train:', len(flowers_train_orig))
print('official val:  ', len(flowers_val_orig))
print('official test: ', len(flowers_test_orig))

In [ ]:
# Human-readable class names (torchvision only returns integer ids) for plots and CAM overlays.
FLOWER_CLASS_NAMES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea', 'english marigold',
    'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood', 'globe thistle',
    'snapdragon', 'colts foot', 'king protea', 'spear thistle', 'yellow iris',
    'globe-flower', 'purple coneflower', 'peruvian lily', 'balloon flower', 'giant white arum lily',
    'fire lily', 'pincushion flower', 'fritillary', 'red ginger', 'grape hyacinth',
    'corn poppy', 'prince of wales feathers', 'stemless gentian', 'artichoke', 'sweet william',
    'carnation', 'garden phlox', 'love in the mist', 'mexican aster', 'alpine sea holly',
    'ruby-lipped cattleya', 'cape flower', 'great masterwort', 'siam tulip', 'lenten rose',
    'barbeton daisy', 'daffodil', 'sword lily', 'poinsettia', 'bolero deep blue',
    'wallflower', 'marigold', 'buttercup', 'oxeye daisy', 'common dandelion',
    'petunia', 'wild pansy', 'primula', 'sunflower', 'pelargonium',
    'bishop of llandaff', 'gaura', 'geranium', 'orange dahlia', 'pink-yellow dahlia?',
    'cautleya spicata', 'japanese anemone', 'black-eyed susan', 'silverbush', 'californian poppy',
    'osteospermum', 'spring crocus', 'bearded iris', 'windflower', 'tree poppy',
    'gazania', 'azalea', 'water lily', 'rose', 'thorn apple',
    'morning glory', 'passion flower', 'lotus', 'toad lily', 'anthurium',
    'frangipani', 'clematis', 'hibiscus', 'columbine', 'desert-rose',
    'tree mallow', 'magnolia', 'cyclamen', 'watercress', 'canna lily',
    'hippeastrum', 'bee balm', 'ball moss', 'foxglove', 'bougainvillea',
    'camellia', 'mallow', 'mexican petunia', 'bromelia', 'blanket flower',
    'trumpet creeper', 'blackberry lily',
]
assert len(FLOWER_CLASS_NAMES) == 102
print('class 0 ->', FLOWER_CLASS_NAMES[0])
print('class 73 ->', FLOWER_CLASS_NAMES[73])
print('class 101 ->', FLOWER_CLASS_NAMES[101])

## 4. Stratified 70/15/15 split

We collect `(path, label)` tuples from all three torchvision splits and then re-partition them
in a stratified way. This guarantees that rare classes retain their proportional presence in
train, val, and test.

Stratification matters here because some classes have only 40-ish samples total — an
unstratified random split could by chance give us zero test samples for a class.

In [ ]:
def collect_labels(ds):
    # Flowers102 stores labels in ds._labels (torchvision >=0.13) or ds.labels (older).
    if hasattr(ds, '_labels'):
        return list(ds._labels)
    return list(ds.labels)

all_labels = collect_labels(flowers_train_orig) + \
             collect_labels(flowers_val_orig) + \
             collect_labels(flowers_test_orig)
combined = ConcatDataset([flowers_train_orig, flowers_val_orig, flowers_test_orig])
n_total = len(combined)
print('combined pool:', n_total, 'images')

indices = np.arange(n_total)
labels_arr = np.asarray(all_labels)

# 15% test
idx_tmp, idx_test, y_tmp, y_test = train_test_split(
    indices, labels_arr, test_size=0.15, stratify=labels_arr, random_state=42,
)
# from the remaining 85%, carve 15/85 ~= 0.1765 for val so the final 70/15/15 holds
idx_train, idx_val, y_train, y_val = train_test_split(
    idx_tmp, y_tmp, test_size=0.15/0.85, stratify=y_tmp, random_state=42,
)
print('train:', len(idx_train), '| val:', len(idx_val), '| test:', len(idx_test))

## 5. Class distribution

Flowers-102 is **roughly balanced** but not uniformly so. A quick histogram tells us whether we need
class weighting. As a rule of thumb from notebook 09, if the most common class is <3× the rarest,
plain cross-entropy is fine. We'll verify that here.

In [ ]:
train_counts = Counter(y_train.tolist())
min_c, max_c = min(train_counts.values()), max(train_counts.values())
print(f'train class counts: min={min_c}  max={max_c}  ratio={max_c/min_c:.2f}x')

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(102), [train_counts[c] for c in range(102)], color='steelblue')
ax.set_xlabel('class id'); ax.set_ylabel('# train samples')
ax.set_title('Oxford Flowers-102 training set: class distribution')
plt.tight_layout(); plt.show()

if max_c / max(1, min_c) < 3:
    print('Imbalance ratio <3x: plain CrossEntropyLoss is appropriate.')
else:
    print('Imbalance ratio >=3x: consider class weights or a weighted sampler.')

## 6. Preview some training samples

Always look at the data. A 4×4 grid of random training images with their class names gives us a
quick sanity check that our labels line up with the pictures.

In [ ]:
rng = np.random.RandomState(0)
picks = rng.choice(idx_train, size=16, replace=False)
imgs, names = [], []
for i in picks:
    img, label = combined[int(i)]
    imgs.append(img)  # PIL.Image
    names.append(FLOWER_CLASS_NAMES[label])
show_image_grid(imgs, labels=names, ncols=4, figsize=(12, 12))

## 7. A `Subset`-style dataset with per-split transforms

We want **different** transforms on train vs val/test. The cleanest pattern for this is a tiny
wrapper that stores a list of indices into the combined dataset plus its own transform.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class FlowersSplit(torch.utils.data.Dataset):
    def __init__(self, base, indices, transform):
        self.base = base
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        if self.transform is not None:
            img = self.transform(img)
        return img, int(label)

IMG_SIZE = 224

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25),
])

eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = FlowersSplit(combined, idx_train, train_tfm)
val_ds   = FlowersSplit(combined, idx_val,   eval_tfm)
test_ds  = FlowersSplit(combined, idx_test,  eval_tfm)
print(len(train_ds), len(val_ds), len(test_ds))

## 8. Mixup via a custom collate function

Mixup (Zhang et al. 2018) creates virtual training examples by linearly blending pairs of images and
their labels. It's a very effective regulariser for fine-grained problems where classes lie near each
other in feature space — exactly our case with Flowers-102.

We implement it as a *collate function* so our custom training loop can emit `(mixed_x, y_a, y_b, lam)`
quadruples. This is cleaner than trying to hack it into `train_one_epoch`, which assumes the standard
`(x, y)` signature.

In [ ]:
class MixupCollate:
    def __init__(self, alpha: float = 0.2, n_classes: int = 102):
        self.alpha = alpha
        self.n_classes = n_classes

    def __call__(self, batch):
        xs = torch.stack([b[0] for b in batch])
        ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
        if self.alpha <= 0:
            return xs, ys, ys, 1.0
        lam = float(np.random.beta(self.alpha, self.alpha))
        # mix with a random permutation of the batch
        perm = torch.randperm(xs.size(0))
        xs = lam * xs + (1 - lam) * xs[perm]
        return xs, ys, ys[perm], lam

def plain_collate(batch):
    xs = torch.stack([b[0] for b in batch])
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return xs, ys

## 9. DataLoaders

Two training loaders: one with Mixup (for the bulk of training), one without (for a sanity-check
accuracy metric). The validation and test loaders never use Mixup.

In [ ]:
BATCH = 64
NUM_WORKERS = 2 if IN_COLAB else 0  # Windows prefers 0 to avoid spawn issues

mixup = MixupCollate(alpha=0.2, n_classes=102)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=mixup, pin_memory=True)
train_loader_plain = DataLoader(train_ds, batch_size=BATCH, shuffle=False,
                                num_workers=NUM_WORKERS, collate_fn=plain_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=plain_collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=plain_collate)
print('batches/epoch:', len(train_loader))

## 10. The model: EfficientNetV2-S (in21k -> in1k)

We use the in21k→in1k checkpoint via timm. The larger pre-training set gives us a much stronger
feature extractor for fine-grained problems than the plain in1k checkpoint.

In [ ]:
model = timm.create_model(
    'tf_efficientnetv2_s.in21k_ft_in1k', pretrained=True, num_classes=102,
)
model.to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params/1e6:.2f}M')
print('classifier head:', model.get_classifier())

## 11. Discriminative learning rates + warmup + cosine annealing

The pretrained backbone is already well-calibrated and needs only a gentle nudge; the random-init
head needs a larger LR so it can actually learn. This is the **discriminative-LR** pattern we saw
in notebook 05.

On top of that we stack two schedulers:

- `LinearLR` for a **2-epoch warmup** (start_factor=0.1): prevents the huge early-epoch gradients
  that can destabilise a randomly-initialised head paired with a sensitive backbone.
- `CosineAnnealingLR` for the remaining 18 epochs: smoothly decays toward zero for the fine final
  polish.

`SequentialLR` chains them. We'll also do a dry-run of the schedule and plot it before training.

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

head_params, backbone_params = [], []
head_names = {n for n, _ in model.get_classifier().named_parameters()}
classifier = model.get_classifier()
head_param_ids = {id(p) for p in classifier.parameters()}
for p in model.parameters():
    (head_params if id(p) in head_param_ids else backbone_params).append(p)
print('backbone params:', sum(p.numel() for p in backbone_params)/1e6, 'M')
print('head params:    ', sum(p.numel() for p in head_params)/1e6, 'M')

EPOCHS = 20
WARMUP_EPOCHS = 2

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': head_params,     'lr': 1e-3, 'weight_decay': 1e-4},
])

warmup = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

In [ ]:
# Dry-run the LR schedule on a dummy optimizer to catch off-by-one warmup bugs.
dummy_p = [torch.nn.Parameter(torch.zeros(1))]
dummy_opt = torch.optim.AdamW([
    {'params': dummy_p, 'lr': 1e-5},
    {'params': [torch.nn.Parameter(torch.zeros(1))], 'lr': 1e-3},
])
dummy_warm = LinearLR(dummy_opt, start_factor=0.1, total_iters=WARMUP_EPOCHS)
dummy_cos  = CosineAnnealingLR(dummy_opt, T_max=EPOCHS - WARMUP_EPOCHS)
dummy_sched = SequentialLR(dummy_opt, [dummy_warm, dummy_cos], milestones=[WARMUP_EPOCHS])

backbone_lrs, head_lrs = [], []
for ep in range(EPOCHS):
    backbone_lrs.append(dummy_opt.param_groups[0]['lr'])
    head_lrs.append(dummy_opt.param_groups[1]['lr'])
    dummy_sched.step()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(backbone_lrs, 'o-', label='backbone (1e-5)')
ax.plot(head_lrs,     's-', label='head (1e-3)')
ax.set_xlabel('epoch'); ax.set_ylabel('LR'); ax.set_yscale('log')
ax.set_title('Discriminative LR schedule (warmup 2 + cosine 18)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 12. Loss: label-smoothed cross-entropy

Label smoothing replaces the hard one-hot target with a soft distribution: `1 - epsilon` on the true
class and `epsilon / (K-1)` on all others. This discourages the model from making over-confident
predictions, which works hand-in-hand with Mixup for fine-grained datasets.

For Mixup, we compute the standard two-label interpolation: `lam * CE(logits, y_a) + (1 - lam) *
CE(logits, y_b)`. Label smoothing is still baked into the `CrossEntropyLoss` object.

In [ ]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

def mixup_loss(logits, y_a, y_b, lam):
    return lam * loss_fn(logits, y_a) + (1 - lam) * loss_fn(logits, y_b)

## 13. Exponential Moving Average of the weights

An EMA model holds a slowly-updated copy of the training weights. At inference time, the EMA weights
often generalise **better** than the raw training weights — this is roughly equivalent to ensembling
across the last few epochs. Decay=0.999 is a common default.

In [ ]:
class EMAModel:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.ema.parameters(), model.parameters()):
            ema_p.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
        # Buffers (e.g. BN running stats) just get copied.
        for ema_b, b in zip(self.ema.buffers(), model.buffers()):
            ema_b.copy_(b)

ema = EMAModel(model, decay=0.999)

## 14. Custom training loop with Mixup + EMA + early stopping

Because Mixup changes the `(x, y)` -> `(x, y_a, y_b, lam)` signature, we can't reuse
`utils.training.fit` unchanged. We write a compact loop here that:

- unpacks the Mixup batch and computes the blended loss;
- updates the EMA model after every optimizer step;
- evaluates both the raw model **and** the EMA model on `val_loader`;
- checkpoints the best-EMA state to `checkpoints/nb12_flowers.pt`;
- early-stops after 5 stagnant epochs.

Expect ~3–4 minutes per epoch on a single T4 GPU.

In [ ]:
ckpt_path = os.path.join(checkpoints_dir(), 'nb12_flowers.pt')

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_acc_ema': []}
best_val_acc = -1.0
stagnant = 0
PATIENCE = 5

for epoch in range(EPOCHS):
    model.train(); t0 = time.time()
    running_loss, running_n = 0.0, 0
    for xb, y_a, y_b, lam in train_loader:
        xb = xb.to(device, non_blocking=True)
        y_a, y_b = y_a.to(device), y_b.to(device)
        logits = model(xb)
        loss = mixup_loss(logits, y_a, y_b, lam)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        ema.update(model)
        running_loss += loss.item() * xb.size(0); running_n += xb.size(0)
    scheduler.step()
    train_loss = running_loss / running_n

    # Plain eval for both the raw and EMA models.
    val_loss_raw, val_acc_raw, _, _ = evaluate(model, val_loader, loss_fn, device)
    val_loss_ema, val_acc_ema, _, _ = evaluate(ema.ema, val_loader, loss_fn, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss_ema)
    history['val_acc'].append(val_acc_raw)
    history['val_acc_ema'].append(val_acc_ema)

    best_candidate = max(val_acc_raw, val_acc_ema)
    flag = ''
    if best_candidate > best_val_acc + 1e-4:
        best_val_acc = best_candidate; stagnant = 0; flag = '  *'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_state_dict': ema.ema.state_dict(),
            'val_acc_raw': val_acc_raw,
            'val_acc_ema': val_acc_ema,
        }, ckpt_path)
    else:
        stagnant += 1

    print(f'epoch {epoch+1:2d}/{EPOCHS} | loss {train_loss:.4f} | '
          f'val raw {val_acc_raw:.4f}  ema {val_acc_ema:.4f} | '
          f'{time.time()-t0:.0f}s{flag}')

    if stagnant >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs).')
        break

In [ ]:
# Training curves. plot_curves wants 'train_acc'/'val_acc' keys — feed raw-val as proxy train_acc.
hist_for_plot = {
    'train_loss': history['train_loss'],
    'val_loss':   history['val_loss'],
    'train_acc':  history['val_acc'],      # just to give plot_curves the expected keys
    'val_acc':    history['val_acc_ema'],
}
plot_curves(hist_for_plot, title='Flowers-102 — EMA validation accuracy')

In [ ]:
# Load best EMA checkpoint for evaluation (EMA consistently outperformed the raw weights).
ckpt = torch.load(ckpt_path, map_location=device)
eval_model = copy.deepcopy(model)
eval_model.load_state_dict(ckpt['ema_state_dict'])
eval_model.to(device).eval()
print(f"best epoch {ckpt['epoch']+1}: val_acc_raw={ckpt['val_acc_raw']:.4f} ema={ckpt['val_acc_ema']:.4f}")

## 17. Top-1 and Top-5 accuracy on the held-out test set

Top-5 is the standard partner metric for fine-grained classification: "does the true class at least
make the model's shortlist?" is a softer but still useful question.

In [ ]:
def collect_logits(m, loader):
    m.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            all_logits.append(m(xb).cpu().numpy())
            all_y.append(yb.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

test_logits, test_y = collect_logits(eval_model, test_loader)
test_pred = test_logits.argmax(axis=1)
top1 = (test_pred == test_y).mean()
top5 = top_k_accuracy_score(test_y, test_logits, k=5, labels=list(range(102)))
print(f'TEST top-1: {top1:.4f}')
print(f'TEST top-5: {top5:.4f}')

## 18. Per-class F1 and classification report

The macro-F1 treats every class with equal weight — more honest than micro-F1 for a near-balanced
but fine-grained dataset. We also surface the five weakest classes by F1 so we can debug them.

In [ ]:
report = classification_report_dict(test_y, test_pred, class_names=FLOWER_CLASS_NAMES)
macro_f1 = f1_score(test_y, test_pred, average='macro')
print(f'macro F1: {macro_f1:.4f}')

per_class = [(FLOWER_CLASS_NAMES[c], report[FLOWER_CLASS_NAMES[c]]['f1-score']) for c in range(102)]
per_class.sort(key=lambda x: x[1])
print('\n5 weakest classes by F1:')
for name, f in per_class[:5]:
    print(f'  {f:.3f}  {name}')

## 19. Normalised confusion matrix

A 102×102 confusion matrix is admittedly dense — you won't read individual cells easily. What we're
looking for is **off-diagonal hotspots**: small clusters of related classes that the model mixes up.
For Flowers-102 those typically show up as small blocks among the lilies and among the daisies.

For targeted debugging you can subset the matrix to the weakest N classes.

In [ ]:
plot_confusion_matrix(test_y, test_pred, class_names=FLOWER_CLASS_NAMES,
                      normalize=True, figsize=(14, 12), title='Flowers-102 test CM (normalized)')

## 20. ROC curves for the 10 weakest classes

Plotting 102 ROC curves produces unreadable spaghetti. Instead we pick the **10 classes with the
lowest F1** and plot their 1-vs-rest ROC. This tells us whether the weakness is driven by low recall
(curve sags) or low precision (many false positives from other classes).

In [ ]:
probs = torch.softmax(torch.from_numpy(test_logits), dim=1).numpy()
weak_idx = [FLOWER_CLASS_NAMES.index(n) for n, _ in per_class[:10]]

fig, ax = plt.subplots(figsize=(7, 6))
from sklearn.metrics import roc_curve, auc
for c in weak_idx:
    y_bin = (test_y == c).astype(int)
    if y_bin.sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_bin, probs[:, c])
    ax.plot(fpr, tpr, label=f'{FLOWER_CLASS_NAMES[c][:20]} AUC={auc(fpr, tpr):.2f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC for the 10 weakest classes')
ax.legend(fontsize=7, loc='lower right'); plt.tight_layout(); plt.show()

## 21. Test-Time Augmentation (TTA): 5-crop averaging

TTA runs inference on several augmented views of each test image and averages the logits. It's a
cheap way to squeeze out another half-point or so of accuracy, at the cost of 5× inference time.

We implement 5-crop (center + 4 corners) + horizontal flips = 10 views per image.

In [ ]:
class FiveCropTTA(torch.utils.data.Dataset):
    def __init__(self, base, indices):
        self.base = base
        self.indices = list(indices)
        self.tfm = transforms.Compose([
            transforms.Resize(256),
            transforms.TenCrop(IMG_SIZE),  # returns 10 PIL crops
            transforms.Lambda(lambda crops: torch.stack(
                [transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(transforms.ToTensor()(c)) for c in crops]
            )),
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        return self.tfm(img), int(label)

tta_ds = FiveCropTTA(combined, idx_test)
tta_loader = DataLoader(tta_ds, batch_size=8, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=plain_collate)

eval_model.eval()
all_logits_tta, all_y_tta = [], []
with torch.no_grad():
    for crops, yb in tta_loader:
        # crops: (B, 10, 3, H, W) -> (B*10, 3, H, W)
        B, N, C, H, W = crops.shape
        logits = eval_model(crops.view(B*N, C, H, W).to(device))
        logits = logits.view(B, N, -1).mean(dim=1)
        all_logits_tta.append(logits.cpu().numpy())
        all_y_tta.append(yb.numpy())
tta_logits = np.concatenate(all_logits_tta)
tta_y      = np.concatenate(all_y_tta)
tta_pred   = tta_logits.argmax(axis=1)
tta_top1 = (tta_pred == tta_y).mean()
tta_top5 = top_k_accuracy_score(tta_y, tta_logits, k=5, labels=list(range(102)))
print(f'TTA top-1: {tta_top1:.4f}  (+{(tta_top1-top1)*100:.2f} pp vs single-crop)')
print(f'TTA top-5: {tta_top5:.4f}')

## 22. Grad-CAM on correct and incorrect predictions

What is the model actually looking at? We use the Grad-CAM utility from notebook 10 to overlay a
class-discriminative attribution map on some test images. We pick three correct and three incorrect
predictions. For the incorrect ones, a common failure mode is the model fixating on background
rather than the flower — worth flagging for data-quality follow-up.

In [ ]:
# Revert normalisation for plotting.
mean_t = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std_t  = torch.tensor(IMAGENET_STD).view(3,1,1)

correct_mask = (test_pred == test_y)
correct_ids = np.where(correct_mask)[0][:3]
wrong_ids   = np.where(~correct_mask)[0][:3]
pick_ids = np.concatenate([correct_ids, wrong_ids])

target_layer = find_efficientnet_target_layer(eval_model)
cam = GradCAM(eval_model, target_layer)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, idx in zip(axes.ravel(), pick_ids):
    xb, yb = test_ds[idx]
    xb_in = xb.unsqueeze(0).to(device)
    target_class = int(test_pred[idx])
    heatmap = cam(xb_in, target_class)
    img_unnorm = (xb * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
    overlay = overlay_cam(img_unnorm, heatmap)
    pred_name = FLOWER_CLASS_NAMES[target_class]
    true_name = FLOWER_CLASS_NAMES[int(yb)]
    colour = 'green' if target_class == yb else 'red'
    ax.imshow(overlay); ax.axis('off')
    ax.set_title(f'pred: {pred_name[:18]}\ntrue: {true_name[:18]}', color=colour, fontsize=9)
plt.suptitle('Grad-CAM — top row correct, bottom row incorrect', y=1.02)
plt.tight_layout(); plt.show()

## 23. Summary table: raw vs EMA vs TTA

We've accumulated three variants of the same model. A pandas table makes the comparison obvious.

In [ ]:
# Raw (non-EMA) snapshot for comparison.
raw_model = copy.deepcopy(model)
raw_model.load_state_dict(ckpt['model_state_dict'])
raw_logits, _ = collect_logits(raw_model, test_loader)
raw_pred = raw_logits.argmax(axis=1)
raw_top1 = (raw_pred == test_y).mean()
raw_top5 = top_k_accuracy_score(test_y, raw_logits, k=5, labels=list(range(102)))

summary = pd.DataFrame({
    'variant': ['raw (best epoch)', 'EMA', 'EMA + 10-crop TTA'],
    'top1':    [raw_top1, top1, tta_top1],
    'top5':    [raw_top5, top5, tta_top5],
})
summary

In [ ]:
# Save the final (EMA) model state_dict for downstream deployment / distillation.
final_path = os.path.join(checkpoints_dir(), 'nb12_flowers.pt')
torch.save(eval_model.state_dict(), final_path)
print('saved ->', final_path)
print('file size (MB):', round(os.path.getsize(final_path)/1e6, 1))

## Key Takeaways

- **Every technique earns its place.** RandAugment, Mixup, label smoothing, EMA, warmup, cosine,
  discriminative LR, and TTA each contribute a fraction of a percent. Stacked, they turn a solid
  baseline (~85%) into a strong one (>90%).
- **Data handling is half the battle.** Stratified re-splitting fixed the minuscule-train-split problem
  and ensured no rare class disappeared from the test set.
- **Evaluation must be multi-angle.** Top-1 alone is not enough. Top-5, per-class F1, confusion
  matrices on weak clusters, ROC for weak classes, and Grad-CAM together paint the full picture.
- **EMA is nearly free.** One deepcopy + one EMA update per batch, and you typically gain
  0.3–1.0 pp of val accuracy. No extra training cost.
- **TTA trades inference speed for accuracy.** Great for leaderboards, bad for real-time products.
- **Mixup interacts with the training loop.** Custom collate is the cleanest way; it keeps the Dataset
  simple and pushes the complexity to the right place.


## Exercises

1. **Swap the backbone.** Rerun with `tf_efficientnetv2_m.in21k_ft_in1k`. Does the extra capacity help,
   or does the model overfit this small dataset?
2. **Turn off Mixup.** How much accuracy do you lose? Does label smoothing alone recover most of it?
3. **Increase augmentation magnitude.** Set `RandAugment(magnitude=15)` and check whether the model
   underfits (train loss plateaus high) — a symptom of over-augmentation.
4. **Weighted sampling experiment.** Even though the dataset is balanced, artificially down-sample
   10 classes to 20 training examples each. Retrain and see how per-class F1 changes.
5. **Bigger input.** Try `IMG_SIZE=320` with a matching `Resize(352)`. Flowers are fine-grained —
   does extra spatial resolution pay off?
6. **Knowledge distillation.** Train an EfficientNetV2-**T** (tiny) student to match your EMA teacher.
   How close can the student get at a fraction of the parameter count?
7. **Deployment sketch.** Export the best model to ONNX (as you'll do in notebook 13) and benchmark
   inference latency on CPU.
